# Caso 2 · Renovaciones en Capesa — Limpieza de datos

Lee el Excel original, corrige sus problemas de calidad y guarda la base limpia (`data/processed/caso2_limpio.pkl`) que usan el diagnóstico y el prototipo.

## 1. Contexto del caso
- **Pregunta:** ¿cómo lograr que el primer contacto vuelva a ser anticipado y suba la renovación con los 5 gestores actuales?
- **Decisión:** qué pólizas se llaman, cuándo y en qué orden; y si hace falta contratar.
- **Éxito:** % de primeros contactos 8–14 días antes del vencimiento (hoy 12 %), renovación en pago manual (hoy 38 %) y comisión por gestor.
- **Restricciones:** 5 gestores, 50 llamadas por día hábil, estándar de contacto a 20 días, ~2,000 vencimientos al mes.
- **Hipótesis:** el contacto tardío baja la renovación y parte de la cartera renueva sola.

## 2. Datos disponibles
Un Excel con cuatro hojas (tabla siguiente). **Faltan y se pedirían:** la regla de apertura de la cola y por qué cambió, canales de contacto registrados (correo/WhatsApp), vigencia de las cotizaciones y calendario de feriados.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Rutas (el notebook corre desde notebooks/caso2/) y parámetros del caso; los lee 01_eda_diagnostico
ROOT = Path.cwd().resolve().parents[1]
assert (ROOT / "pyproject.toml").is_file(), (
    f"Ejecuta el notebook desde notebooks/caso2/ (raíz detectada: {ROOT})"
)
DATA_CASE2 = ROOT / "data" / "raw" / "caso2_renovaciones.xlsx"
PROCESSED_CASE2 = ROOT / "data" / "processed" / "caso2_limpio.pkl"
PARAMS = {
    "staffing_change_date": pd.Timestamp("2025-12-01"),  # salida de G06 y G07 (hoja Equipo)
    "daily_call_target": 50,  # meta de llamadas por gestor y día
    "standard_lead_days": 20,  # estándar: contactar 20 días antes del vencimiento
}
STAFFING_CHANGE_DATE = PARAMS["staffing_change_date"]
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

# Parámetros de limpieza (ver §3 Supuestos)
EFFECTIVE_RESULTS = ["Contactado", "Pide llamar después", "Cotización enviada", "Cierre"]  # S2.12 · ver §3
SHARED_PHONE_MAX_CLIENTS = 5  # S2.3 · ver §3
LEAD_TIME_BINS = [
    -99,
    -1,
    3,
    7,
    14,
    20,
    25,
    31,
]  # S2.13 · ver §3 (pd.cut cierra a la derecha: "0-3" = {0,1,2,3})
LEAD_TIME_LABELS = ["Después del venc.", "0-3", "4-7", "8-14", "15-20", "21-25", "26-30"]
PERIOD_LABELS = {7: "7 gestores (sep-nov 25)", 5: "5 gestores (dic 25 →)"}

raw = pd.read_excel(DATA_CASE2, sheet_name=None)
pd.DataFrame({k: v.shape for k, v in raw.items()}, index=["filas", "columnas"]).T

,filas,columnas
Diccionario,34,3
Renovaciones,25351,20
Bitacora_Llamadas,78058,6
Equipo,13,8


## 3. Supuestos
Decisiones que condicionan los resultados de este notebook y por qué se tomaron. En el texto y en el código se citan por su código.

| Código | Supuesto | Por qué | Dónde se aplica |
|---|---|---|---|
| **S2.1** | Las filas duplicadas exactas de *Renovaciones* se eliminan. | Son 151 copias de la misma póliza; duplicarían vencimientos y comisión. | §4.2 |
| **S2.2** | *Renovó* (Sí/SI/No/no) se normaliza a 1/0; cualquier otra grafía detiene la carga. | Son variantes de captura del mismo valor. Detener la carga evita clasificar en silencio un valor nuevo en ciclos futuros. | §4.2 |
| **S2.3** | Un teléfono es válido si cumple `####-####` y no lo comparten más de 5 clientes distintos (`SHARED_PHONE_MAX_CLIENTS`); si no, la póliza no es contactable por teléfono. | Dos números se repiten en 79 y 86 clientes distintos: son relleno. Ninguna de las 522 pólizas sin teléfono válido logró un contacto efectivo. | §4.2 |
| **S2.4** | Una *Fecha primer contacto* nula se completa con la primera llamada de la bitácora. | Cuando ambas existen coinciden en el 100 % de los casos. | §4.4 |
| **S2.5** | Las 222 pólizas renovadas sin llamada de *Cierre* conservan su resultado, pero se excluyen de los análisis que relacionan llamadas con renovación. | *Renovó* es el dato oficial, pero ninguna tuvo un contacto efectivo y todas tienen teléfono inválido: su renovación no se puede atribuir a la gestión telefónica. | §4.4 |
| **S2.6** | Un primer contacto después del vencimiento es un contacto tardío real. | Las fechas coinciden con la bitácora; son el extremo del problema que se estudia (579 pólizas), no un error de captura. | §4.4 |
| **S2.7** | Dos periodos según la fecha de vencimiento: *7 gestores* (antes del 1-dic-2025) y *5 gestores* (desde el 1-dic-2025) (`staffing_change_date`). | G06 y G07 salen el 30-nov-2025 según la hoja *Equipo*; la fecha de vencimiento define con qué equipo se gestionó la póliza. | §4.1 |
| **S2.8** | **Anticipación** = días entre el primer **intento** de contacto y el vencimiento. | Así define el diccionario *Fecha primer contacto*. La anticipación al primer contacto atendido (`lead_days_effective`) queda disponible para sensibilidad. | §4.4 |
| **S2.9** | **Comisión potencial** = prima de renovación × % comisión; comisión ganada = potencial × renovó. | Es lo que Capesa cobra si la póliza renueva: mide el valor de cada póliza y el costo de perderla. | §4.3 |
| **S2.12** | Una llamada es efectiva si su resultado es *Contactado*, *Pide llamar después*, *Cotización enviada* o *Cierre* (`EFFECTIVE_RESULTS`). | En esos resultados se habló con el cliente; *No contesta* y *Número equivocado* no son contacto. | §4.4 |
| **S2.13** | Tramos de anticipación: después del vencimiento, 0–3, 4–7, 8–14, 15–20, 21–25 y 26–30 días (`LEAD_TIME_BINS`). | Separan el estándar de 20 días, la ventana de 8–14 días que más renueva y el contacto de último momento (≤ 3 días). | §4.4 |

## 4. Validación y limpieza

### 4.1 Equipo y dotación
La hoja *Equipo* trae dos tablas apiladas: los gestores y la dotación por periodo.

In [2]:
sheet = raw["Equipo"]
first_col = sheet.iloc[:, 0]
header_row = sheet.index[first_col.astype(str).str.strip() == "Periodo"][0]
team = sheet.iloc[: sheet.index[first_col.isna()][0]].copy()
staffing = sheet.iloc[header_row + 1 :].copy()
staffing.columns = sheet.iloc[header_row].tolist()
staffing = staffing.dropna(axis=1, how="all")

display(
    team[
        [
            "Código",
            "Nombre",
            "Fecha de ingreso al área",
            "Fecha de salida",
            "Estado",
            "Meta llamadas diarias",
            "Meta tasa de renovación",
        ]
    ]
)
display(staffing)

,Código,Nombre,Fecha de ingreso al área,Fecha de salida,Estado,Meta llamadas diarias,Meta tasa de renovación
0,G01,Ernesto Ramón Altamirano,2019-03-04 00:00:00,NaN,Activo,50.00,0.70
1,G02,Luis Fernando Gaitán,2021-08-16 00:00:00,NaN,Activo,50.00,0.70
2,G03,Karla Vanessa Úbeda,2017-01-09 00:00:00,NaN,Activo,50.00,0.70
3,G04,Martha Elena Zeledón,2022-05-02 00:00:00,NaN,Activo,50.00,0.70
4,G05,Ana Lucía Morales,2023-11-06 00:00:00,NaN,Activo,50.00,0.70
5,G06,Roberto Carlos Membreño,2020-02-10 00:00:00,2025-11-30 00:00:00,Retirado,50.00,0.70
6,G07,Silvia Patricia Obando,2022-09-19 00:00:00,2025-11-30 00:00:00,Retirado,50.00,0.70


,Periodo,Gestores,Códigos,Comentario
11,01/09/2025 - 30/11/2025,7,G01 a G07,Trimestre con dotación completa
12,01/12/2025 - a la fecha,5,G01 a G05,Salida de G06 y G07; carteras redistribuidas


Siete gestores con la misma meta (50 llamadas/día, 70 % de renovación). **G06 y G07 salen el 30-nov-2025** y el análisis se divide en dos periodos: *7 gestores* (sep–nov 2025) y *5 gestores* (desde dic 2025).

*Supuesto S2.7 (ver §3).*

### 4.2 Duplicados, *Renovó* y teléfonos

In [3]:
ren_raw = raw["Renovaciones"]
print(
    f"Filas duplicadas exactas: {ren_raw.duplicated().sum()} (= pólizas repetidas: {ren_raw['Póliza'].duplicated().sum()}; "
    "no hay pólizas repetidas con datos distintos)"
)
print("Valores de 'Renovó' tal como vienen:", ren_raw["Renovó"].value_counts().to_dict())

r = ren_raw.drop_duplicates().copy()  # S2.1 · ver §3
r["renewed"] = r["Renovó"].str.strip().str.lower().map({"sí": 1, "si": 1, "no": 0})  # S2.2 · ver §3
assert r["renewed"].notna().all(), (
    f"Valor de 'Renovó' no reconocido: {r.loc[r['renewed'].isna(), 'Renovó'].unique()}"
)

# S2.3 · ver §3
r["phone_format_ok"] = r["Teléfono cliente"].str.fullmatch(r"\d{4}-\d{4}").fillna(False).astype(bool)
clientes_por_tel = r.groupby("Teléfono cliente")["ID cliente"].nunique()
r["shared_phone"] = r["Teléfono cliente"].map(clientes_por_tel).gt(SHARED_PHONE_MAX_CLIENTS).fillna(False)
r["valid_phone"] = r["phone_format_ok"] & ~r["shared_phone"]

print("\nTeléfonos que no cumplen el formato ####-####:")
print(r.loc[~r["phone_format_ok"], "Teléfono cliente"].fillna("<nulo>").value_counts().to_string())
print(
    "\nTeléfonos con formato válido pero compartidos por más de",
    SHARED_PHONE_MAX_CLIENTS,
    "clientes distintos:",
)
print(
    clientes_por_tel[
        clientes_por_tel.gt(SHARED_PHONE_MAX_CLIENTS) & clientes_por_tel.index.str.fullmatch(r"\d{4}-\d{4}")
    ].to_string()
)
print(f"\nPólizas no contactables por teléfono: {(~r['valid_phone']).sum()} de {len(r)}")

Filas duplicadas exactas: 151 (= pólizas repetidas: 151; no hay pólizas repetidas con datos distintos)
Valores de 'Renovó' tal como vienen: {'Sí': 13463, 'No': 11762, 'SI': 70, 'no': 56}

Teléfonos que no cumplen el formato ####-####:
Teléfono cliente
N/D         83
<nulo>      71
0888888     70
0452-123    68
505         65

Teléfonos con formato válido pero compartidos por más de 5 clientes distintos:
Teléfono cliente
0041-9610    86
0186-7825    79

Pólizas no contactables por teléfono: 522 de 25200


- 151 duplicados exactos se eliminan.
- *Renovó* viene en cuatro grafías y se normaliza a 1/0.
- **522 pólizas no son contactables por teléfono:** 357 con formato inválido y 165 con dos números de relleno compartidos por decenas de clientes. En §4.4 se ve que ninguna logró contacto.

*Supuestos S2.1, S2.2, S2.3 (ver §3).*

### 4.3 Nulos y consistencia interna

In [4]:
print("Nulos en Renovaciones (tras quitar duplicados):")
print(r[ren_raw.columns].isna().sum()[lambda s: s > 0].to_string())
chk = (r["Prima de renovación (USD)"] / r["Prima anterior (USD)"] - 1 - r["% cambio de prima"]).abs()
print(f"\n'% cambio de prima' inconsistente con las primas: {(chk > 0.01).sum()}")
print(
    f"'Motivo de no renovación' nulo en pólizas NO renovadas: {(r['Motivo de no renovación'].isna() & (r['renewed'] == 0)).sum()}"
)
print(
    f"'Fecha primer contacto' nula: {r['Fecha primer contacto'].isna().sum()} (se resuelve con la bitácora en §4.4)"
)

r["expiry_month"] = r["Fecha de vencimiento"].dt.to_period("M")
r["potential_commission"] = r["Prima de renovación (USD)"] * r["% comisión"]  # S2.9 · ver §3
r["earned_commission"] = r["potential_commission"] * r["renewed"]
r["active_agents"] = np.where(r["Fecha de vencimiento"] < STAFFING_CHANGE_DATE, 7, 5)  # S2.7 · ver §3
r["period"] = r["active_agents"].map(PERIOD_LABELS)

Nulos en Renovaciones (tras quitar duplicados):
Teléfono cliente              71
Fecha primer contacto        151
Motivo de no renovación    13453

'% cambio de prima' inconsistente con las primas: 0
'Motivo de no renovación' nulo en pólizas NO renovadas: 0
'Fecha primer contacto' nula: 151 (se resuelve con la bitácora en §4.4)


Nulos esperables: el motivo de no renovación solo existe si no se renovó y las primas cuadran con el % de cambio. Faltan 151 fechas de primer contacto, que se completan con la bitácora (§4.4).

*Supuesto S2.9 (ver §3).*

### 4.4 Bitácora de llamadas y coherencia con Renovaciones
¿Coinciden los intentos y la fecha de primer contacto con la bitácora?

*Supuestos S2.12, S2.13 (ver §3).*

In [5]:
calls = raw["Bitacora_Llamadas"].copy()
calls["date"] = calls["Fecha y hora"].dt.normalize()
calls["month"] = calls["Fecha y hora"].dt.to_period("M")
calls["hour"] = calls["Fecha y hora"].dt.hour
calls["is_effective"] = calls["Resultado"].isin(EFFECTIVE_RESULTS)  # S2.12 · ver §3
calls = calls.sort_values("Fecha y hora")
calls["attempt_no"] = calls.groupby("Póliza").cumcount() + 1
print("Resultados de llamada:", calls["Resultado"].value_counts().to_dict())
print(f"Empates de fecha y hora dentro de una póliza: {calls.duplicated(['Póliza', 'Fecha y hora']).sum()}")

# Una fila por póliza con lo que dice la bitácora
effective = calls[calls["is_effective"]]
by_policy = calls.groupby("Póliza")
resumen = pd.DataFrame(
    {
        "calls_logged": by_policy.size(),
        "first_call": by_policy["date"].min(),
        "first_effective_contact": effective.groupby("Póliza")["date"].min(),
        "attempt_first_effective": effective.groupby("Póliza")["attempt_no"].min(),
        "had_closing_call": by_policy["Resultado"].agg(lambda s: (s == "Cierre").any()),
        "last_result": by_policy["Resultado"].last(),
    }
)
r = r.merge(resumen, left_on="Póliza", right_index=True, how="left")
r["first_contact_missing"] = r["Fecha primer contacto"].isna()
r["Fecha primer contacto"] = r["Fecha primer contacto"].fillna(r["first_call"])  # S2.4 · ver §3
r["lead_days"] = (r["Fecha de vencimiento"] - r["Fecha primer contacto"]).dt.days  # S2.8 · ver §3
r["lead_days_effective"] = (r["Fecha de vencimiento"] - r["first_effective_contact"]).dt.days
r["lead_bucket"] = pd.cut(r["lead_days"], LEAD_TIME_BINS, labels=LEAD_TIME_LABELS)  # S2.13 · ver §3
r["renewed_without_closing"] = (r["renewed"] == 1) & ~r["had_closing_call"].fillna(False).astype(
    bool
)  # S2.5 · ver §3

print(
    f"\nIDs de llamada duplicados: {calls['ID llamada'].duplicated().sum()} | llamadas a pólizas que no existen: {(~calls['Póliza'].isin(r['Póliza'])).sum()} "
    f"| pólizas sin ninguna llamada: {r['calls_logged'].isna().sum()}"
)
print(
    f"Llamadas en fin de semana: {(calls['Fecha y hora'].dt.day_of_week >= 5).sum()} | fuera de 8-17h: {(~calls['hour'].between(8, 16)).sum()}"
)
print(f"'Número de intentos' ≠ llamadas en bitácora: {(r['Número de intentos'] != r['calls_logged']).sum()}")
otro_gestor = calls.merge(r[["Póliza", "Gestor"]], on="Póliza", suffixes=("", "_poliza")).eval(
    "Gestor != Gestor_poliza"
)
print(
    f"Llamadas hechas por un gestor distinto al asignado a la póliza: {otro_gestor.sum()} ({otro_gestor.mean():.2%}; se mantienen, cuentan como trabajo del equipo)"
)
print(
    "'Fecha primer contacto' ≠ primera llamada en bitácora (excluyendo los nulos completados): "
    f"{((r['Fecha primer contacto'] != r['first_call']) & ~r['first_contact_missing']).sum()}"
)
print(
    f"Primer contacto nulo: {r['first_contact_missing'].sum()} → completado con la primera llamada de la bitácora"
)
print(
    f"Anticipación fuera de los tramos definidos (NaN): {r['lead_bucket'].isna().sum()}; rango de lead_days: {r['lead_days'].min()} a {r['lead_days'].max()}"
)
print(
    f"Primer contacto DESPUÉS del vencimiento: {(r['lead_days'] < 0).sum()} ({(r['lead_days'] < 0).mean():.1%})"
)

sin_cierre = r[r["renewed_without_closing"]]
print(
    f"\nRenovó = Sí sin llamada de 'Cierre': {len(sin_cierre)} | de ellas con algún contacto efectivo: {sin_cierre['attempt_first_effective'].notna().sum()} "
    f"| con teléfono inválido: {(~sin_cierre['valid_phone']).sum()} | última llamada: {sin_cierre['last_result'].value_counts().to_dict()}"
)
print(
    f"'Cierre' en bitácora pero Renovó = No: {((r['renewed'] == 0) & r['had_closing_call'].fillna(False)).sum()}"
)
print(
    f"Pólizas con teléfono inválido que lograron contacto efectivo: {(~r['valid_phone'] & r['attempt_first_effective'].notna()).sum()} "
    f"| llamadas gastadas en ellas: {r.loc[~r['valid_phone'], 'calls_logged'].sum():.0f} ({r.loc[~r['valid_phone'], 'calls_logged'].sum() / len(calls):.1%} de la bitácora)"
)

Resultados de llamada: {'No contesta': 21305, 'Cotización enviada': 20292, 'Cierre': 13231, 'Pide llamar después': 10833, 'Contactado': 10744, 'Número equivocado': 1653}
Empates de fecha y hora dentro de una póliza: 0



IDs de llamada duplicados: 0 | llamadas a pólizas que no existen: 0 | pólizas sin ninguna llamada: 0
Llamadas en fin de semana: 0 | fuera de 8-17h: 0
'Número de intentos' ≠ llamadas en bitácora: 0
Llamadas hechas por un gestor distinto al asignado a la póliza: 30 (0.04%; se mantienen, cuentan como trabajo del equipo)
'Fecha primer contacto' ≠ primera llamada en bitácora (excluyendo los nulos completados): 0
Primer contacto nulo: 151 → completado con la primera llamada de la bitácora
Anticipación fuera de los tramos definidos (NaN): 0; rango de lead_days: -12 a 30
Primer contacto DESPUÉS del vencimiento: 579 (2.3%)

Renovó = Sí sin llamada de 'Cierre': 222 | de ellas con algún contacto efectivo: 0 | con teléfono inválido: 222 | última llamada: {'Número equivocado': 139, 'No contesta': 83}
'Cierre' en bitácora pero Renovó = No: 0
Pólizas con teléfono inválido que lograron contacto efectivo: 0 | llamadas gastadas en ellas: 2401 (3.1% de la bitácora)


La bitácora **cuadra** con Renovaciones: intentos y fecha de primer contacto coinciden, así que los 151 nulos se completan sin riesgo.
- **222 pólizas renovadas sin llamada de cierre** tienen todas teléfono inválido: se respeta *Renovó*, pero se excluyen de los análisis de llamadas.
- **579 primeros contactos después del vencimiento** son el problema que se estudia, no un error.
- Las 522 pólizas con teléfono inválido no lograron ningún contacto y consumieron el 3 % de las llamadas.

*Supuestos S2.4, S2.5, S2.6, S2.8 (ver §3).*

## 5. Base limpia y exportación
Se guarda en `data/processed/caso2_limpio.pkl`: una fila por póliza con el resumen de la bitácora, el periodo, la comisión y el tramo de anticipación, más la bitácora, el equipo y los parámetros.

In [6]:
calidad = pd.DataFrame(
    [
        ("Renovaciones", "Filas duplicadas exactas", ren_raw.duplicated().sum(), "Eliminadas (S2.1)"),
        (
            "Renovaciones",
            "'Renovó' con 4 grafías (Sí/SI/No/no)",
            (~ren_raw["Renovó"].isin(["Sí", "No"])).sum(),
            "Normalizado a 1/0 (S2.2)",
        ),
        (
            "Renovaciones",
            "Teléfono nulo, de relleno o con formato inválido",
            int((~r["phone_format_ok"]).sum()),
            "Marcado como no contactable (S2.3)",
        ),
        (
            "Renovaciones",
            "Teléfono con formato válido compartido por decenas de clientes",
            int((r["phone_format_ok"] & r["shared_phone"]).sum()),
            "Marcado como no contactable (S2.3)",
        ),
        (
            "Renovaciones",
            "Fecha de primer contacto nula",
            int(r["first_contact_missing"].sum()),
            "Completada con la primera llamada de la bitácora (S2.4)",
        ),
        (
            "Renovaciones",
            "Renovó = Sí sin llamada de 'Cierre' ni contacto efectivo",
            int(r["renewed_without_closing"].sum()),
            "Se respeta 'Renovó'; se excluye de análisis llamada → renovación (S2.5)",
        ),
        (
            "Renovaciones",
            "Primer contacto después del vencimiento",
            int((r["lead_days"] < 0).sum()),
            "Se mantiene: contacto tardío real (S2.6)",
        ),
        ("Bitácora", "IDs duplicados, pólizas inexistentes, llamadas fuera de horario", 0, "Sin tratamiento"),
        ("Ambas", "Intentos y primer contacto vs bitácora", "0 dif.", "Cuadra: sin tratamiento"),
    ],
    columns=["Hoja", "Problema", "Filas", "Tratamiento"],
)
display(calidad.style.hide(axis="index"))

base_limpia = {
    "renewals": r,
    "calls": calls,
    "team": team,
    "staffing": staffing,
    "quality": calidad,
    "effective_results": EFFECTIVE_RESULTS,
    "lead_time_bins": LEAD_TIME_BINS,
    "lead_time_labels": LEAD_TIME_LABELS,
    "params": PARAMS,
}
destino = PROCESSED_CASE2
destino.parent.mkdir(parents=True, exist_ok=True)
pd.to_pickle(base_limpia, destino)
print(
    f"Base limpia escrita en {destino.relative_to(ROOT)}: "
    + ", ".join(f"{k} ({len(v)})" for k, v in base_limpia.items() if isinstance(v, pd.DataFrame))
)

Hoja,Problema,Filas,Tratamiento
Renovaciones,Filas duplicadas exactas,151,Eliminadas (S2.1)
Renovaciones,'Renovó' con 4 grafías (Sí/SI/No/no),126,Normalizado a 1/0 (S2.2)
Renovaciones,"Teléfono nulo, de relleno o con formato inválido",357,Marcado como no contactable (S2.3)
Renovaciones,Teléfono con formato válido compartido por decenas de clientes,165,Marcado como no contactable (S2.3)
Renovaciones,Fecha de primer contacto nula,151,Completada con la primera llamada de la bitácora (S2.4)
Renovaciones,Renovó = Sí sin llamada de 'Cierre' ni contacto efectivo,222,Se respeta 'Renovó'; se excluye de análisis llamada → renovación (S2.5)
Renovaciones,Primer contacto después del vencimiento,579,Se mantiene: contacto tardío real (S2.6)
Bitácora,"IDs duplicados, pólizas inexistentes, llamadas fuera de horario",0,Sin tratamiento
Ambas,Intentos y primer contacto vs bitácora,0 dif.,Cuadra: sin tratamiento


Base limpia escrita en data/processed/caso2_limpio.pkl: renewals (25200), calls (78058), team (7), staffing (2), quality (9)


**Qué se exporta:** `renewals` (una fila por póliza con columnas derivadas como `renewed`, `valid_phone`, `lead_days`, `lead_bucket`, `potential_commission`, `period`), `calls`, `team`, `staffing` y `params`. Se lee con `pd.read_pickle`.